# example how to generate the WEED standard feature cube based on EO data (Sentinel-1 and Sentinel-2)
this tests include the generation of the feature cube with and without NVBT band. The NVBT band is a measure of the number of valid input data timesteps after cloud masking and the internal temporal binning.
The NVBT band can be useful for identifying areas with high or low levels of observation, which can be important for a variety of applications, such as monitoring changes in land use or assessing the quality of data in a given area.

In [1]:
from eo_processing.utils.helper import init_connection
from eo_processing.openeo.processing import generate_master_feature_cube
from eo_processing.config.settings import get_advanced_options, get_job_options, get_collection_options

#### declare space and time

In [2]:
# the time context is given by start and end date
year = 2024
start = f'{year}-01-01'
end = f'{year+1}-01-01'   # the end is always exclusive

# the space context is defined as a bounding box dictionary with south,west,north,east and crs
# we take as example an 10x10km tile in EU LAEA grid around Vienna
AOI = {'west': 4780000, 'east': 4790000, 'south': 2830000, 'north': 2840000, 'crs': 3035}

### get processing_options for the eo_processing functions, collection_options and job_options

In [3]:
processing_options = get_advanced_options(provider='cdse', skip_check_S1=False, skip_check_S2=True)
job_options = get_job_options(provider='cdse', task='feature_generation')
collection_options = get_collection_options(provider='cdse')
processing_options.update({'openeo_chunk_size': 64})

In [4]:
processing_options

{'provider': 'cdse',
 's1_orbitdirection': 'DESCENDING',
 'target_crs': 3035,
 'resolution': 10.0,
 'time_interpolation': False,
 'ts_interval': 'dekad',
 'S2_temporal_reducer': 'median',
 'S1_temporal_reducer': 'mean',
 'SLC_masking_algo': 'mask_scl_dilation',
 'S2_max_cloud_cover': 95,
 'S2_bands': ['B02',
  'B03',
  'B04',
  'B05',
  'B06',
  'B07',
  'B08',
  'B8A',
  'B11',
  'B12'],
 's2_tileid_list': None,
 'skip_check_S1': False,
 'skip_check_S2': True,
 'apply_cloud_mask': True,
 'get_NVBT': False,
 'optical_vi_list': ['ABDI1',
  'ABDI2',
  'AWEInsh',
  'AVI',
  'BLFEI',
  'CIRE',
  'EVI',
  'IRECI',
  'MBWI',
  'MNDWI',
  'MNDVI',
  'NDMI',
  'NDVI',
  'NDVIMNDWI',
  'NDWI',
  'NMDI',
  'NIRv',
  'S2WI',
  'S2REP',
  'WRI'],
 'radar_vi_list': ['VHVVD', 'VHVVR', 'DpRVIVV'],
 'S2_scaling': [0, 10000, 0, 1.0],
 'S1_db_rescale': True,
 'append': True,
 'openeo_chunk_size': 64}

### establish connection to openEO

In [5]:
con = init_connection(provider='cdse')

Authenticated using refresh token.


### run the feature cube generation WITHOUT NVBT band

In [6]:
# update job_options due toBerts setting from last inference runs
# ToDO: optimize the job settings for feature_cube_generation_with_nobs & feature_cube_generation and put in settings + add correct task to 'get_job_options'
job_options.update({
    "driver-memory": "4G",
    "driver-memoryOverhead": "4G",
    "executor-memory": "5G",
    "executor-memoryOverhead": "3g",
    "max-executors": 10,
    "python-memory": "disable",
    "allow_empty_cubes": True,
    "soft-errors": 0.05})

In [ ]:
# get master cube without nobs
data = generate_master_feature_cube(con, AOI, start, end, **collection_options, **processing_options)

In [ ]:
data.execute_batch(r'C:\Users\buchhorm\Downloads\test_cube\features_cube_v5.tif', title='feature without nobs (10x10km)', job_options=job_options)

## run with activated NVBT generation

In [ ]:
# now we run same with nobs_perc band
processing_options.update({'get_NVBT': True})
data2 = generate_master_feature_cube(con, AOI, start, end, **collection_options, **processing_options)

In [ ]:
data2.execute_batch(r'C:\Users\buchhorm\Downloads\test_cube\features_cube_with_nobs_v5.tif', title='feature with nobs (10x10km)', job_options=job_options)

## now we run this 10x10km test in a bigger context - we generate the full EO feature cube with NVBT band
Note: since the new ECDC version and alphaEarth STACs are not complete THIS loading has to be tested later

In [7]:
from habitat_mapping.openeo.feature_cubes import create_EOfeature_cube_WEED_V1
processing_options.update(target_crs = 3035)
processing_options.update({'get_NVBT': True})
processing_options.update({'openeo_chunk_size': 16})
job_options.update({"allow_empty_cubes": True})
job_options.update({
    "driver-memory": "4G",
    "driver-memoryOverhead": "4G",
    "executor-memory": "5G",
    "executor-memoryOverhead": "3G",
    "max-executors": 10,
    "python-memory": "disable",
    "soft-errors": 0.05,
})
data3 = create_EOfeature_cube_WEED_V1(con, AOI, start, end, collection_options, processing_options)

Deriving band listing from unordered `item_assets`
The specified bands ['precipitation-flux', 'temperature-mean'] in `load_stac` are not a subset of the bands [] found in the STAC metadata (unknown bands: ['precipitation-flux', 'temperature-mean']). Working with specified bands as is.


In [8]:
data3.execute_batch(r'C:\Users\buchhorm\Downloads\test_cube\eo_feature_cube_beta2_10x10.tif', title='eo_feature cube (10x10km)', job_options=job_options)

0:00:00 Job 'j-2607011012174934aeec7870e2685f99': send 'start'
0:00:07 Job 'j-2607011012174934aeec7870e2685f99': queued (progress 0%)
0:00:13 Job 'j-2607011012174934aeec7870e2685f99': queued (progress 0%)
0:00:19 Job 'j-2607011012174934aeec7870e2685f99': queued (progress 0%)
0:00:27 Job 'j-2607011012174934aeec7870e2685f99': queued (progress 0%)
0:00:37 Job 'j-2607011012174934aeec7870e2685f99': queued (progress 0%)
0:00:50 Job 'j-2607011012174934aeec7870e2685f99': queued (progress 0%)
0:01:05 Job 'j-2607011012174934aeec7870e2685f99': running (progress 8.9%)
0:01:24 Job 'j-2607011012174934aeec7870e2685f99': running (progress 11.5%)
0:01:48 Job 'j-2607011012174934aeec7870e2685f99': running (progress 14.5%)
0:02:19 Job 'j-2607011012174934aeec7870e2685f99': running (progress 18.0%)
0:02:56 Job 'j-2607011012174934aeec7870e2685f99': running (progress 22.0%)
0:03:43 Job 'j-2607011012174934aeec7870e2685f99': running (progress 26.5%)
0:04:41 Job 'j-2607011012174934aeec7870e2685f99': running (pro

<BatchJob job_id='j-2607011012174934aeec7870e2685f99'>

## now upscale the the final 20x20km tiles

In [ ]:
# we take as example an 20x20km tile in EU LAEA grid around Vienna
AOI = {'west': 4780000, 'east': 4800000, 'south': 2820000, 'north': 2840000, 'crs': 3035}
# increasing driver memory
job_options.update({
    "driver-memory": "10G",
    "driver-memoryOverhead": "4G",
})
data4 = create_EOfeature_cube_WEED_V1(con, AOI, start, end, collection_options, processing_options)

In [ ]:
data4.execute_batch(r'C:\Users\buchhorm\Downloads\test_cube\eo_feature_cube_beta2_20x20.tif', title='eo_feature cube (20x20km)', job_options=job_options)